# 03 - Data Preprocessing & Feature Engineering

## Objective

The objective of this notebook is to transform the raw Ames Housing dataset into a clean, numerical, and model-ready representation for regression.

The preprocessing workflow includes:

- Separating independent variables and the target variable.
- Creating training and validation datasets.
- Handling missing numerical and categorical values.
- Encoding categorical features using One-Hot Encoding.
- Scaling numerical features using StandardScaler.
- Building a reusable preprocessing pipeline.
- Preventing data leakage by fitting transformations exclusively on the training data.
- Validating the final feature matrices.

The same processed data will be used to train Scikit-learn's Linear Regression model and our custom Batch, Stochastic, and Mini-Batch Gradient Descent implementations.

The original raw datasets will remain unchanged.

In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
TARGET = "SalePrice"
VALIDATION_SIZE = 0.20

PROJECT_ROOT = Path.cwd().parent

TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.csv"

train_df = pd.read_csv(TRAIN_PATH)

print("Dataset Shape:", train_df.shape)

train_df.head()

Dataset Shape: (1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [5]:
X = train_df.drop(columns=[TARGET, "Id"])
y = train_df[TARGET]

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Matrix Shape: (1460, 79)
Target Shape: (1460,)


In [6]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=VALIDATION_SIZE, random_state=RANDOM_STATE)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

X_train: (1168, 79)
X_val: (292, 79)
y_train: (1168,)
y_val: (292,)


In [7]:
numerical_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(include="string").columns.tolist()

print("Numerical Features:", len(numerical_features))
print("Categorical Features:", len(categorical_features))

Numerical Features: 36
Categorical Features: 43


In [8]:
print("Numerical Columns:")
print(numerical_features)

print("\nCategorical Columns:")
print(categorical_features)

Numerical Columns:
['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold']

Categorical Columns:
['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType',

In [9]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [10]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

In [11]:
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
], remainder="drop", verbose_feature_names_out=False)

In [12]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

In [13]:
y_train_array = y_train.to_numpy(dtype=np.float64)

y_val_array = y_val.to_numpy(dtype=np.float64)

In [14]:
X_train_processed = np.asarray(X_train_processed, dtype=np.float64)
X_val_processed = np.asarray(X_val_processed, dtype=np.float64)

In [19]:
print("Processed Training Shape:", X_train_processed.shape)
print("Processed Validation Shape:", X_val_processed.shape)

print("Training Target Shape:", y_train_array.shape)
print("Validation Target Shape:", y_val_array.shape)

Processed Training Shape: (1168, 301)
Processed Validation Shape: (292, 301)
Training Target Shape: (1168,)
Validation Target Shape: (292,)


In [28]:
feature_names = preprocessor.get_feature_names_out()

print("Number of Transformed Features: ", feature_names)

print(feature_names[:30])

Number of Transformed Features:  ['MSSubClass' 'LotFrontage' 'LotArea' 'OverallQual' 'OverallCond'
 'YearBuilt' 'YearRemodAdd' 'MasVnrArea' 'BsmtFinSF1' 'BsmtFinSF2'
 'BsmtUnfSF' 'TotalBsmtSF' '1stFlrSF' '2ndFlrSF' 'LowQualFinSF'
 'GrLivArea' 'BsmtFullBath' 'BsmtHalfBath' 'FullBath' 'HalfBath'
 'BedroomAbvGr' 'KitchenAbvGr' 'TotRmsAbvGrd' 'Fireplaces' 'GarageYrBlt'
 'GarageCars' 'GarageArea' 'WoodDeckSF' 'OpenPorchSF' 'EnclosedPorch'
 '3SsnPorch' 'ScreenPorch' 'PoolArea' 'MiscVal' 'MoSold' 'YrSold'
 'MSZoning_C (all)' 'MSZoning_FV' 'MSZoning_RH' 'MSZoning_RL'
 'MSZoning_RM' 'Street_Grvl' 'Street_Pave' 'Alley_Grvl' 'Alley_Missing'
 'Alley_Pave' 'LotShape_IR1' 'LotShape_IR2' 'LotShape_IR3' 'LotShape_Reg'
 'LandContour_Bnk' 'LandContour_HLS' 'LandContour_Low' 'LandContour_Lvl'
 'Utilities_AllPub' 'Utilities_NoSeWa' 'LotConfig_Corner'
 'LotConfig_CulDSac' 'LotConfig_FR2' 'LotConfig_FR3' 'LotConfig_Inside'
 'LandSlope_Gtl' 'LandSlope_Mod' 'LandSlope_Sev' 'Neighborhood_Blmngtn'
 'Neighborhoo

In [21]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_val_processed_df = pd.DataFrame(
    X_val_processed,
    columns=feature_names,
    index=X_val.index
)

X_train_processed_df.head()

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
254,-0.866764,-0.012468,-0.212896,-0.820445,0.372217,-0.455469,-1.346063,-0.597889,1.037269,-0.285504,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1066,0.074110,-0.502357,-0.265245,-0.088934,1.268609,0.718609,0.439214,-0.597889,-0.971996,-0.285504,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
638,-0.631546,-0.146074,-0.177841,-0.820445,1.268609,-1.988293,-1.683818,-0.597889,-0.971996,-0.285504,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
799,-0.161109,-0.457822,-0.324474,-0.820445,1.268609,-1.107734,-1.683818,0.861522,0.267995,-0.285504,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
380,-0.161109,-0.903175,-0.529035,-0.820445,0.372217,-1.531707,-1.683818,-0.597889,-0.496920,-0.285504,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [22]:
print("Missing Values in Training: ", np.isnan(X_train_processed).sum())
print("Missing Values in Validation: ", np.isnan(X_val_processed).sum())

Missing Values in Training:  0
Missing Values in Validation:  0


In [23]:
print("Infinite Values in Training: ", np.isinf(X_train_processed).sum())
print("Infinite Values in Validation: ", np.isinf(X_val_processed).sum())

Infinite Values in Training:  0
Infinite Values in Validation:  0


In [24]:
print("Training Target NaNs: ", np.isnan(y_train_array).sum())
print("Validation Target NaNs: ", np.isnan(y_val_array).sum())

print("Training Target Infinities: ", np.isinf(y_train_array).sum())
print("Validation Target Infinities: ", np.isinf(y_val_array).sum())

Training Target NaNs:  0
Validation Target NaNs:  0
Training Target Infinities:  0
Validation Target Infinities:  0


In [25]:
processed_summary = pd.DataFrame({
    "Mean": X_train_processed_df.mean(),
    "Std": X_train_processed_df.std(ddof=0),
    "Min": X_train_processed_df.min(),
    "Max": X_train_processed_df.max()
})

processed_summary.head(20)

,Mean,Std,Min,Max
MSSubClass,6.995926e-17,1.0,-0.866764,3.131951
LotFrontage,-2.007527e-16,1.0,-2.194699,10.809613
LotArea,2.281280e-17,1.0,-0.873069,19.019985
OverallQual,-5.170902e-17,1.0,-3.746488,2.837110
OverallCond,-2.281280e-16,1.0,-4.109739,3.061391
YearBuilt,-1.417435e-15,1.0,-3.227597,1.273035
YearRemodAdd,4.653812e-15,1.0,-1.683818,1.211225
MasVnrArea,-4.562560e-18,1.0,-0.597889,7.382542
BsmtFinSF1,6.463627e-18,1.0,-0.971996,11.327666
BsmtFinSF2,-4.410475e-17,1.0,-0.285504,6.840653


In [26]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

np.save(PROCESSED_DIR / "X_train.npy", X_train_processed)
np.save(PROCESSED_DIR / "X_val.npy", X_val_processed)

np.save(PROCESSED_DIR / "y_train.npy", y_train_array)

np.save(PROCESSED_DIR / "y_val.npy", y_val_array)

np.save(PROCESSED_DIR / "feature_names.npy", feature_names)

print("Processed datasets saved successfully")

Processed datasets saved successfully


## Conclusion

The Ames Housing dataset was successfully transformed into a numerical representation suitable for regression modeling.

A reproducible preprocessing pipeline was developed using Scikit-learn's `Pipeline` and `ColumnTransformer`.

Numerical features were processed using median imputation and StandardScaler, while categorical features were handled using explicit missing-category imputation and One-Hot Encoding.

The dataset was divided into training and validation partitions before fitting preprocessing transformations, preventing information leakage from the validation data.

The transformed feature matrices were validated for missing values, infinite values, and dimensional consistency.

The resulting datasets are now ready for baseline regression modeling and the implementation of custom Gradient Descent optimization algorithms.

The fitted preprocessing pipeline will be reused across subsequent models to ensure a consistent and fair comparison.